# RNN Basics

Vanilla recurrent networks and backpropagation through time (BPTT) — implemented from
scratch, validated against PyTorch, and stress-tested on sequence tasks that expose
temporal structure and the vanishing-gradient problem.

We answer four concrete questions with numbers:
1. What does a vanilla RNN compute, and how does BPTT propagate gradients?
2. Do analytical gradients match `nn.RNN` to machine precision?
3. How quickly do gradient magnitudes decay across timesteps when the loss is at the end?
4. Can an RNN learn temporal patterns that a shuffled-time baseline cannot?

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

np.random.seed(0)
plt.rcParams['figure.dpi'] = 100

## 1. From-Scratch Vanilla RNN

A vanilla RNN maintains a hidden state $h_t$ updated at each timestep:

$$h_t = \tanh\big(x_t W_{xh} + h_{t-1} W_{hh} + b\big)$$

We use **batch-first** layout: input $x$ has shape $(N, T, D)$.
Weights follow the repo convention: $W_{xh}$ is $(D, H)$, $W_{hh}$ is $(H, H)$,
and each step computes `x_t @ W_xh + h @ W_hh + b`.

**BPTT** unrolls the network over $T$ steps and applies the chain rule backward in time.
The key term $W_{hh}^\top$ is applied at every timestep — if its spectral radius $< 1$,
gradients **vanish**; if $> 1$, they **explode**.

In [2]:
def tanh(x):
    return np.tanh(x)


def tanh_grad(x):
    return 1.0 - np.tanh(x) ** 2


class VanillaRNN:
    """Batch-first vanilla RNN with tanh activation and full BPTT."""

    def __init__(self, input_size, hidden_size, seed=0):
        self.input_size, self.hidden_size = input_size, hidden_size
        rng = np.random.RandomState(seed)
        self.W_xh = rng.randn(input_size, hidden_size) * np.sqrt(2.0 / input_size)
        self.W_hh = rng.randn(hidden_size, hidden_size) * np.sqrt(2.0 / hidden_size)
        self.b = np.zeros(hidden_size)
        self.cache = None

    def forward(self, x):
        N, T, D = x.shape
        hs = np.zeros((N, T, self.hidden_size), dtype=x.dtype)
        h = np.zeros((N, self.hidden_size), dtype=x.dtype)
        zs, h_prevs = [], [h.copy()]
        for t in range(T):
            z = x[:, t] @ self.W_xh + h @ self.W_hh + self.b
            h = tanh(z)
            zs.append(z)
            hs[:, t] = h
            h_prevs.append(h.copy())
        self.cache = (x, np.stack(zs, axis=1), h_prevs)
        return hs, h

    def backward(self, dhs):
        x, zs, h_prevs = self.cache
        N, T, D = x.shape
        dW_xh = np.zeros_like(self.W_xh)
        dW_hh = np.zeros_like(self.W_hh)
        db = np.zeros_like(self.b)
        dx = np.zeros_like(x)
        dh = np.zeros((N, self.hidden_size), dtype=x.dtype)
        dh_norms = []
        for t in reversed(range(T)):
            dh += dhs[:, t]
            dh_norms.append(float(np.linalg.norm(dh)))
            dz = dh * tanh_grad(zs[:, t])
            dW_xh += x[:, t].T @ dz
            dW_hh += h_prevs[t].T @ dz
            db += dz.sum(axis=0)
            dx[:, t] = dz @ self.W_xh.T
            dh = dz @ self.W_hh.T
        dh_norms.reverse()
        return dx, dW_xh, dW_hh, db, dh_norms

print('VanillaRNN defined.')

VanillaRNN defined.


## 2. Validation Against PyTorch

Forward pass and BPTT compared to `nn.RNN(batch_first=True, nonlinearity='tanh')` in float64.
PyTorch stores `weight_ih` as $(H, D)$ — our $W_{xh}^\top$.

In [3]:
rng42 = np.random.RandomState(42)
N, T, D, H = 4, 6, 3, 5
x_np = rng42.randn(N, T, D).astype(np.float64)
rnn = VanillaRNN(D, H, seed=99)
rnn.W_xh = rng42.randn(D, H).astype(np.float64) * 0.2
rnn.W_hh = rng42.randn(H, H).astype(np.float64) * 0.2
rnn.b = rng42.randn(H).astype(np.float64) * 0.01

hs, _ = rnn.forward(x_np)
dhs = rng42.randn(N, T, H).astype(np.float64)
dx, dW_xh, dW_hh, db, _ = rnn.backward(dhs)

x_t = torch.tensor(x_np, requires_grad=True, dtype=torch.float64)
layer = nn.RNN(D, H, batch_first=True, nonlinearity='tanh', bias=True, num_layers=1).double()
with torch.no_grad():
    layer.weight_ih_l0.copy_(torch.tensor(rnn.W_xh.T))
    layer.weight_hh_l0.copy_(torch.tensor(rnn.W_hh.T))
    layer.bias_ih_l0.copy_(torch.tensor(rnn.b))
    layer.bias_hh_l0.zero_()
out_t, _ = layer(x_t)
(out_t * torch.tensor(dhs)).sum().backward()

print(f"Forward max diff: {np.abs(hs - out_t.detach().numpy()).max():.2e}")
print(f"dx max diff:      {np.abs(dx - x_t.grad.numpy()).max():.2e}")
print(f"dW_xh max diff:   {np.abs(dW_xh - layer.weight_ih_l0.grad.numpy().T).max():.2e}")
print(f"dW_hh max diff:   {np.abs(dW_hh - layer.weight_hh_l0.grad.numpy().T).max():.2e}")
print(f"db max diff:      {np.abs(db - layer.bias_ih_l0.grad.numpy()).max():.2e}")

Forward max diff: 1.11e-16
dx max diff:      1.11e-16
dW_xh max diff:   1.78e-15
dW_hh max diff:   6.66e-16
db max diff:      8.88e-16


## 3. Vanishing Gradients Through Time

Place a unit impulse at $t=0$ and backpropagate a loss that depends **only on the final
hidden state** ($t=T-1$). The gradient $\partial L / \partial h_t$ at early timesteps is
repeatedly multiplied by $W_{hh}^\top$ and $\tanh'(z)$ — shrinking when $|W_{hh}| < 1$.

In [4]:
def grad_norm_profile(T=40, hidden=16, whh_scale=0.9):
    rnn = VanillaRNN(1, hidden, seed=0)
    rnn.W_hh = np.eye(hidden) * whh_scale
    rnn.W_xh *= 0.1
    x = np.zeros((1, T, 1), dtype=np.float64)
    x[0, 0, 0] = 1.0
    rnn.forward(x)
    dhs = np.zeros((1, T, hidden))
    dhs[0, -1, 0] = 1.0
    *_, norms = rnn.backward(dhs)
    return norms

T = 40
norms_09 = grad_norm_profile(T, 16, 0.9)
norms_05 = grad_norm_profile(T, 16, 0.5)
print(f"W_hh = 0.9 I:  ||dh|| at t=0 = {norms_09[0]:.4e}, t={T-1} = {norms_09[-1]:.4e}, ratio t0/T = {norms_09[0]/norms_09[-1]:.4e}")
print(f"W_hh = 0.5 I:  ratio t0/T = {norms_05[0]/norms_05[-1]:.4e}")

fig, ax = plt.subplots(figsize=(8, 4))
ax.semilogy(range(T), norms_09, '-o', ms=3, label='W_hh = 0.9 I')
ax.semilogy(range(T), norms_05, '-o', ms=3, label='W_hh = 0.5 I')
ax.set_xlabel('timestep t')
ax.set_ylabel('||dL/dh_t|| (log scale)')
ax.set_title('Gradient magnitude through BPTT (loss at final timestep)')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('bptt_grad_norms.png', dpi=100, bbox_inches='tight')
plt.show()

W_hh = 0.9 I:  ||dh|| at t=0 = 1.2969e-02, t=39 = 1.0000e+00, ratio t0/T = 1.2969e-02
W_hh = 0.5 I:  ratio t0/T = 1.7833e-12


C:\Users\PM_IntellicaBD\AppData\Local\Temp\ipykernel_2132\1301985765.py:28: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Sequence Classification: Temporal Pattern Task

Each sequence is length $T=20$. **Class 0** has high values in the **first half**;
**Class 1** in the **second half**. The label is read from the final hidden state
(many-to-one RNN classifier).

We compare against a flattened MLP baseline on the same data.

In [5]:
def relu(x):
    return np.maximum(0, x)


def relu_grad(x):
    return (x > 0).astype(x.dtype)


def softmax(z):
    z = z - z.max(axis=1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)


class RNNClassifier:
    def __init__(self, input_size, hidden_size, n_classes=2, seed=0):
        self.rnn = VanillaRNN(input_size, hidden_size, seed=seed)
        rng = np.random.RandomState(seed + 1)
        self.W_hy = rng.randn(hidden_size, n_classes) * np.sqrt(2.0 / hidden_size)
        self.b_y = np.zeros(n_classes)

    def forward(self, x):
        hs, h = self.rnn.forward(x)
        return hs, h @ self.W_hy + self.b_y

    def step(self, x, y, lr=0.05):
        hs, logits = self.forward(x)
        probs = softmax(logits)
        n = len(y)
        loss = -np.mean(np.log(probs[np.arange(n), y] + 1e-12))
        dlogits = probs.copy()
        dlogits[np.arange(n), y] -= 1
        dlogits /= n
        h_last = self.rnn.cache[2][-1]
        dW_hy = h_last.T @ dlogits
        db_y = dlogits.sum(axis=0)
        dhs = np.zeros_like(hs)
        dhs[:, -1] = dlogits @ self.W_hy.T
        dx, dW_xh, dW_hh, db, _ = self.rnn.backward(dhs)
        self.W_hy -= lr * dW_hy
        self.b_y -= lr * db_y
        self.rnn.W_xh -= lr * dW_xh
        self.rnn.W_hh -= lr * dW_hh
        self.rnn.b -= lr * db
        return loss

    def predict(self, x):
        return self.forward(x)[1].argmax(axis=1)


class MLPSeq:
    def __init__(self, seq_len, input_size, hidden=64, n_classes=2, seed=0):
        rng = np.random.RandomState(seed)
        d = seq_len * input_size
        self.W1 = rng.randn(d, hidden) * np.sqrt(2.0 / d)
        self.b1 = np.zeros(hidden)
        self.W2 = rng.randn(hidden, n_classes) * np.sqrt(2.0 / hidden)
        self.b2 = np.zeros(n_classes)

    def forward(self, x):
        flat = x.reshape(x.shape[0], -1)
        self.z1 = flat @ self.W1 + self.b1
        self.a1 = relu(self.z1)
        return self.a1 @ self.W2 + self.b2

    def step(self, x, y, lr=0.01):
        logits = self.forward(x)
        probs = softmax(logits)
        n = len(y)
        loss = -np.mean(np.log(probs[np.arange(n), y] + 1e-12))
        dlogits = probs.copy()
        dlogits[np.arange(n), y] -= 1
        dlogits /= n
        dW2 = self.a1.T @ dlogits
        db2 = dlogits.sum(axis=0)
        da1 = dlogits @ self.W2.T
        dz1 = da1 * relu_grad(self.z1)
        flat = x.reshape(x.shape[0], -1)
        dW1 = flat.T @ dz1
        db1 = dz1.sum(axis=0)
        self.W2 -= lr * dW2
        self.b2 -= lr * db2
        self.W1 -= lr * dW1
        self.b1 -= lr * db1
        return loss

    def predict(self, x):
        return self.forward(x).argmax(axis=1)


def make_pattern_data(n=800, T=20, seed=0):
    rng = np.random.RandomState(seed)
    X = rng.randn(n, T, 1) * 0.3
    y = rng.randint(0, 2, size=n)
    for i in range(n):
        if y[i] == 0:
            X[i, :T // 2, 0] += 2.0
        else:
            X[i, T // 2:, 0] += 2.0
    idx = rng.permutation(n)
    split = int(0.75 * n)
    tr, te = idx[:split], idx[split:]
    return X[tr], y[tr], X[te], y[te]


X_train, y_train, X_test, y_test = make_pattern_data()
train_rng = np.random.RandomState(0)

rnn_clf = RNNClassifier(1, 32, seed=0)
for epoch in range(60):
    idx = train_rng.permutation(len(y_train))
    for i in range(0, len(idx), 32):
        b = idx[i:i + 32]
        rnn_clf.step(X_train[b], y_train[b], lr=0.05)
    if (epoch + 1) % 20 == 0:
        acc = (rnn_clf.predict(X_test) == y_test).mean()
        print(f"Epoch {epoch+1:>2} (RNN): test acc={acc:.3f}")

train_rng = np.random.RandomState(0)
mlp_clf = MLPSeq(20, 1, seed=0)
for epoch in range(60):
    idx = train_rng.permutation(len(y_train))
    for i in range(0, len(idx), 32):
        b = idx[i:i + 32]
        mlp_clf.step(X_train[b], y_train[b], lr=0.01)
    if (epoch + 1) % 20 == 0:
        acc = (mlp_clf.predict(X_test) == y_test).mean()
        print(f"Epoch {epoch+1:>2} (MLP): test acc={acc:.3f}")

rnn_acc = (rnn_clf.predict(X_test) == y_test).mean()
mlp_acc = (mlp_clf.predict(X_test) == y_test).mean()
print(f"\nFinal: RNN={rnn_acc:.3f}, MLP={mlp_acc:.3f}")

Epoch 20 (RNN): test acc=1.000


Epoch 40 (RNN): test acc=1.000


Epoch 60 (RNN): test acc=1.000
Epoch 20 (MLP): test acc=1.000
Epoch 40 (MLP): test acc=1.000
Epoch 60 (MLP): test acc=1.000

Final: RNN=1.000, MLP=1.000


## 5. Order Matters: Shuffled Timesteps

If temporal order is destroyed by permuting timesteps within each sequence, the RNN cannot
detect which half had high values. Both models should fall to ~chance — but the RNN is
explicitly built to exploit order.

In [6]:
def shuffle_time(X, seed=99):
    rng = np.random.RandomState(seed)
    out = np.empty_like(X)
    for i in range(len(X)):
        out[i] = X[i, rng.permutation(X.shape[1])]
    return out

X_test_shuf = shuffle_time(X_test)
rnn_shuf = (rnn_clf.predict(X_test_shuf) == y_test).mean()
mlp_shuf = (mlp_clf.predict(X_test_shuf) == y_test).mean()
print(f"Original order:  RNN={rnn_acc:.3f}, MLP={mlp_acc:.3f}")
print(f"Shuffled timesteps: RNN={rnn_shuf:.3f}, MLP={mlp_shuf:.3f}")

Original order:  RNN=1.000, MLP=1.000
Shuffled timesteps: RNN=0.445, MLP=0.520


## 6. Long-Range Dependency Stress Test

**Delayed recall**: the first timestep encodes the label ($+1$ or $-1$), followed by
$T-1$ steps of noise. The network must carry information across the full sequence.
With $T=100$ and a smaller hidden size, vanilla RNN training becomes unreliable — a
preview of why gated architectures (LSTM/GRU, Topic 11) were developed.

In [7]:
def make_delay_data(n=800, T=30, seed=0):
    rng = np.random.RandomState(seed)
    X = rng.randn(n, T, 1) * 0.5
    y = (rng.rand(n) > 0.5).astype(int)
    X[:, 0, 0] = np.where(y, 1.0, -1.0)
    idx = rng.permutation(n)
    split = int(0.75 * n)
    tr, te = idx[:split], idx[split:]
    return X[tr], y[tr], X[te], y[te]


def train_delay(T, hidden=16, epochs=150, lr=0.05, seed=0):
    Xtr, ytr, Xte, yte = make_delay_data(n=800, T=T, seed=seed)
    model = RNNClassifier(1, hidden, seed=seed)
    rng = np.random.RandomState(0)
    for _ in range(epochs):
        idx = rng.permutation(len(ytr))
        for i in range(0, len(idx), 32):
            b = idx[i:i + 32]
            model.step(Xtr[b], ytr[b], lr=lr)
    return (model.predict(Xte) == yte).mean()

print(f"{'T':>6} {'seed=0':>8} {'seed=1':>8} {'seed=2':>8} {'mean':>8}")
for T in [10, 50, 100]:
    accs = [train_delay(T, hidden=16, epochs=150 if T <= 50 else 200, lr=0.05 if T <= 50 else 0.01, seed=s)
            for s in range(3)]
    print(f"{T:>6} {accs[0]:>8.3f} {accs[1]:>8.3f} {accs[2]:>8.3f} {np.mean(accs):>8.3f}")

     T   seed=0   seed=1   seed=2     mean


    10    1.000    1.000    1.000    1.000


    50    0.475    0.450    0.490    0.472


   100    0.530    0.550    0.535    0.538
